## ⚠️ 本地数据已存在 — 跳过此 notebook

你的股票数据已存储在本地硬盘（挂载后路径为 `/Users/karan/Desktop/20260320/`）。

所有后续 notebook 已改为直接从本地 CSV 读取，**无需运行此 notebook**。

直接从 `01_getting_started.ipynb` 开始即可。

# 00 — 数据准备
## 一次性运行：下载并缓存全市场价格宽表

这个 notebook **只需跑一次**。它会：
1. 构建股票池（默认：中证500）
2. 批量下载所有股票的日线数据（前复权收盘价）
3. 构建价格宽表（date × symbol）并保存到 `data/processed/`

之后所有研究 notebook 都直接加载缓存，无需重复下载。

---

**预计耗时**：
- 中证500（500只 × 5年）：约 5-10 分钟
- 沪深300（300只 × 5年）：约 3-6 分钟
- 全A股（5000只）：约 1-2 小时，不建议首次直接跑

In [ ]:
# import sys
# sys.path.insert(0, "../../..")

# import warnings
# warnings.filterwarnings("ignore")

# import pandas as pd
# import matplotlib.pyplot as plt

# from utils.universe import build_universe, filter_st
# from utils.data_loader import build_price_matrix, build_return_matrix

# print("✅ 模块导入成功")

## 1. 构建股票池

选择研究范围。建议从 **中证500** 开始：
- 300只太少，1000只太慢，500只正好
- 中证500覆盖中盘股，比300更能发现 alpha
- 流动性充足，不会出现数据缺失严重的问题

In [ ]:
# 选择股票池
# 可选: 'csi300' | 'csi500' | 'csi1000' | 'sse50' | 'all'
# UNIVERSE_MODE = "csi500"

# symbols = build_universe(UNIVERSE_MODE)

# 过滤 ST 股（不参与正常交易，会污染因子信号）
# symbols = filter_st(symbols)

# print(f"最终股票池: {len(symbols)} 只")
# print(f"前10只: {symbols[:10]}")

## 2. 下载数据并构建宽表

价格宽表格式：
```
         000001  000002  000006  ...
2019-01-02  9.21   27.10   4.50  ...
2019-01-03  9.05   26.88   4.45  ...
...
```

数据会缓存到 `data/processed/`，下次直接读缓存（秒级）。

In [ ]:
# START = "2019-01-01"
# END   = "2024-12-31"

# 下载并缓存（首次运行较慢，之后秒读缓存）
# price_wide = build_price_matrix(
#     symbols,
#     start=START,
#     end=END,
#     col="close",
#     max_workers=4,    # 并发线程数，akshare 建议 ≤ 5
# )

# print(f"\n价格宽表形状: {price_wide.shape}")
# print(f"日期范围: {price_wide.index[0].date()} → {price_wide.index[-1].date()}")
# print(f"交易日数: {len(price_wide)}")
# price_wide.iloc[:3, :5]  # 预览

## 3. 检查数据质量

In [ ]:
# 每只股票的数据完整度
# total_days = len(price_wide)
# coverage = price_wide.notna().sum() / total_days

# print("数据完整度分布:")
# print(coverage.describe().round(3))
# print(f"\n完整度 > 95% 的股票数: {(coverage > 0.95).sum()}")
# print(f"完整度 < 50% 的股票数: {(coverage < 0.50).sum()}  （可能是新上市或退市）")

# fig, ax = plt.subplots(figsize=(10, 4))
# ax.hist(coverage, bins=40, color="steelblue", alpha=0.7, edgecolor="white")
# ax.axvline(0.95, color="red", linestyle="--", label="95% 完整度")
# ax.set_title("股票数据完整度分布")
# ax.set_xlabel("有效交易日占比")
# ax.set_ylabel("股票数量")
# ax.legend()
# plt.tight_layout()
# plt.show()

In [ ]:
# 每个交易日的有效股票数（截面覆盖率）
# daily_coverage = price_wide.notna().sum(axis=1)

# fig, ax = plt.subplots(figsize=(13, 4))
# ax.fill_between(daily_coverage.index, daily_coverage, alpha=0.5, color="teal")
# ax.set_title("每日有效股票数")
# ax.set_ylabel("有效股票数")
# ax.set_ylim(0, len(symbols) * 1.1)
# plt.tight_layout()
# plt.show()

# print(f"平均每日覆盖: {daily_coverage.mean():.0f} 只 / {len(symbols)} 只 ({daily_coverage.mean()/len(symbols):.1%})")

## 4. 构建收益率宽表

In [ ]:
# ret_wide = build_return_matrix(price_wide)

# print(f"收益率宽表形状: {ret_wide.shape}")

# 截面收益率分布（用所有股票所有日期的收益率）
# all_rets = ret_wide.values.flatten()
# all_rets = all_rets[~pd.isna(all_rets)]
# 去掉涨跌停极值（±11%）
# all_rets_clipped = all_rets[(all_rets > -0.11) & (all_rets < 0.11)]

# fig, ax = plt.subplots(figsize=(10, 4))
# ax.hist(all_rets_clipped, bins=100, color="steelblue", alpha=0.7, edgecolor="white", density=True)
# ax.axvline(0, color="black", linewidth=0.8)
# ax.set_title(f"全市场日收益率分布（{len(all_rets_clipped):,} 个观测，已去掉±11%极值）")
# ax.set_xlabel("日收益率")
# ax.set_ylabel("密度")
# plt.tight_layout()
# plt.show()

## 5. 同步下载基准指数

In [ ]:
# from utils.data_loader import get_index_history, calc_returns

# 沪深300 作为基准
# hs300 = get_index_history("sh000300", START, END)
# hs300_ret = calc_returns(hs300["close"])

# 中证500
# zz500 = get_index_history("sh000905", START, END)
# zz500_ret = calc_returns(zz500["close"])

# print(f"沪深300: {len(hs300)} 个交易日")
# print(f"中证500: {len(zz500)} 个交易日")

# 画累计收益
# fig, ax = plt.subplots(figsize=(12, 4))
# cum_300 = (1 + hs300_ret).cumprod()
# cum_500 = (1 + zz500_ret).cumprod()
# ax.plot(cum_300.index, cum_300, label="沪深300", linewidth=1.5)
# ax.plot(cum_500.index, cum_500, label="中证500", linewidth=1.5)
# ax.axhline(1, color="gray", linewidth=0.5, linestyle="--")
# ax.set_title(f"基准指数累计收益（{START} — {END}）")
# ax.set_ylabel("净值")
# ax.legend()
# plt.tight_layout()
# plt.show()

---

## 数据准备完成 ✅

| 产出 | 位置 | 说明 |
|------|------|------|
| 个股数据 | `data/raw/*.parquet` | 每只股票单独缓存 |
| 价格宽表 | `data/processed/price_wide_*.parquet` | date × symbol |
| 基准指数 | 内存（无需缓存，随时拉取快） | 沪深300、中证500 |

### 在其他 notebook 中加载数据

```python
from utils.data_loader import build_price_matrix, build_return_matrix
from utils.universe import build_universe

symbols = build_universe('csi500')  # 直接读缓存
price_wide = build_price_matrix(symbols, '2019-01-01', '2024-12-31')  # 秒级加载
ret_wide   = build_return_matrix(price_wide)
```

### 下一步

→ `01_getting_started.ipynb`：单股分析入门  
→ `02_statistics_basics.ipynb`：统计检验，判断信号是否显著